# Chapter 04: What's in an Action: uses, Marketplace, Versioning (notebook edition)

## Learning Objectives

- Classify uses: references
- Audit tag-to-SHA resolution from a log
- Explain why a floating tag can be missing

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Audit the runner's log

The runner logs which commit each action resolved to. We parse the saved log lines from run 37312325910; a tag and its SHA should map to one commit.

In [ ]:
import json, re
forms = json.loads((ROOT/"fixtures"/"run_ch04_forms.json").read_text())
LINE = re.compile(r"Download action repository '([^']+)' \(SHA:([0-9a-f]{40})\)")
resolved = {m.group(1): m.group(2) for l in forms["download_log_lines"] if (m := LINE.search(l))}
print(resolved)
assert len(resolved) == 2 and len(set(resolved.values())) == 1

## 2. Classify the five reference forms

We load the demo workflow and classify each `uses:` value.

In [ ]:
from intro_gha import WORKFLOWS_DIR
from intro_gha.workflow import load_workflow
def classify(ref):
    if ref.startswith("./"): return "local"
    if ref.startswith("docker://"): return "docker"
    v = ref.partition("@")[2]
    return "sha" if len(v) == 40 and all(c in "0123456789abcdef" for c in v) else "tag-or-branch"
wf = load_workflow(WORKFLOWS_DIR / "ch04-uses-forms.yml")
uses = [s["uses"] for s in wf["jobs"]["forms"]["steps"] if "uses" in s]
print([classify(u) for u in uses])
assert [classify(u) for u in uses] == ["tag-or-branch", "sha", "local", "docker"]

## 3. Floating tags and runtimes

The tags fixture records where each checkout tag pointed on 2026-10-05, and which Node runtime each version declares.

In [ ]:
tags = json.loads((ROOT/"fixtures"/"action_tags_2026_10.json").read_text())
co = tags["actions/checkout"]
print(co["tags"]["v7"], co["tags"]["v7.0.1"], co["tags"]["v7.0.0"])
assert co["tags"]["v7"] == co["tags"]["v7.0.1"] != co["tags"]["v7.0.0"]
assert (co["runtime"]["v4"], co["runtime"]["v7"]) == ("node20", "node24")

## 4. The unresolvable tag

Run 37312465693 died in `Set up job`; no step of ours ran.

In [ ]:
bad = json.loads((ROOT/"fixtures"/"run_ch04_badref.json").read_text())
print(bad["error"])
assert bad["failed_step"] == "Set up job" and bad["steps_run"] == 1

## Takeaways & Next Steps

- A tag resolves to a commit every run, and the log shows which.
- Only a SHA is immutable; floating tags may not even exist.
- Next: Chapter 05, the machines that run all this.

## Chapter Link

Read: `learning_modules/chapter_04_whats_in_an_action.md`